# 00 Preprocessing

This notebook preprocesses the event logs used in the following experiments: the originals are read
from `logs/raw/`, and every log the experiments use is written to `logs/` reduced to the case id, the
activity and the timestamp.


In [1]:
import re

import pandas as pd
import pm4py

RAW_DIR = "logs/raw"  # the untouched originals
OUT_DIR = "logs"  # the reduced copies the experiments read
KEEP = ["case:concept:name", "concept:name", "time:timestamp"]


def save(df, name):
    """Write one log to logs/, reduced to the three columns the experiments use."""
    out = df[KEEP]
    pm4py.write_xes(out, f"{OUT_DIR}/{name}.xes", case_id_key="case:concept:name")
    print(f"{name}.xes: {out['case:concept:name'].nunique()} cases, {len(out)} events")




  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




# BPIChallenge 2011 (Hospital)

Split on the median age into `age_low` and `age_high`.


In [2]:
bpic11 = pm4py.read_xes(f"{RAW_DIR}/BPIC11.xes")

# Age repeats once per Diagnosis-Treatment Combination (Age, Age:1, ... Age:5) and about
# half the cases have no unsuffixed Age at all -> take the earliest age on record. The
# copies differ by at most 3 years within a case, so the choice barely matters.
age_cols = [c for c in bpic11.columns if re.fullmatch(r"case:Age(:\d+)?", c)]
bpic11["case:age"] = bpic11[age_cols].min(axis=1)

bpic11_median = bpic11.groupby("case:concept:name")["case:age"].first().median()
print(f"median age: {bpic11_median}  (from {len(age_cols)} Age columns)")

bpic11["case:age_group"] = (bpic11["case:age"] > bpic11_median).map(
    {True: "age_high", False: "age_low"}
)

save(bpic11, "BPIC11")
for label, df in bpic11.groupby("case:age_group"):
    save(df, f"BPIC11_{label}")


/Users/felixgross/Documents/Python Projects/Embedding Comparison/.venv/lib/python3.14/site-packages/pm4py/utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(


median age: 58.0  (from 6 Age columns)
BPIC11.xes: 1143 cases, 150291 events
BPIC11_age_high.xes: 540 cases, 67363 events
BPIC11_age_low.xes: 603 cases, 82928 events


# BPIChallenge 2015

Five municipalities, five separate logs — each kept on its own, plus the first two merged into the
pair `BPIC15_M1_M2`. Case ids repeat across municipalities, so each is prefixed with its
municipality before the merge.

The activity is taken from `activityNameEN`, not from `concept:name`: the latter holds the
internal activity code, which splits one activity into several numbered variants
(`01_HOOFD_065_1`, `01_HOOFD_065_2`, ...). `activityNameEN` is the readable label those codes
share, and it is the alphabet the experiments are meant to compare on — 272 activities instead of
356 for M4, with 69 labels covering more than one code.


In [3]:
bpic15 = []
for i in range(1, 6):
    df = pm4py.read_xes(f"{RAW_DIR}/BPIC15_M{i}.xes")
    df["case:concept:name"] = f"M{i}_" + df["case:concept:name"].astype(str)
    # the readable label, shared by the numbered variants of one activity code
    df["concept:name"] = df["activityNameEN"]
    save(df, f"BPIC15_M{i}")
    if i <= 2:  # only the first pair is merged; the rest need not stay in memory
        bpic15.append(df)

save(pd.concat(bpic15, ignore_index=True), "BPIC15_M1_M2")


BPIC15_M1.xes: 1199 cases, 52217 events
BPIC15_M2.xes: 832 cases, 44354 events
BPIC15_M3.xes: 1409 cases, 59681 events
BPIC15_M4.xes: 1053 cases, 47293 events
BPIC15_M5.xes: 1156 cases, 59083 events
BPIC15_M1_M2.xes: 2031 cases, 96571 events


# BPIChallenge 2018

Split on the department that handled the application, one log per department, plus the first two
merged into a pair.


In [4]:
bpic18 = pm4py.read_xes(f"{RAW_DIR}/BPIC18.xes")

bpic18["case:department"] = bpic18.groupby("case:concept:name")[
    "case:department"
].transform("first")

for dep, df in bpic18.groupby("case:department"):
    save(df, f"BPIC18_D{dep}")

first_two = sorted(bpic18["case:department"].unique())[:2]
save(
    bpic18[bpic18["case:department"].isin(first_two)],
    "BPIC18_D" + "_D".join(first_two),
)


BPIC18_D4e.xes: 13639 cases, 735363 events
BPIC18_D6b.xes: 11256 cases, 648709 events
BPIC18_Dd4.xes: 5744 cases, 393265 events
BPIC18_De7.xes: 13170 cases, 736929 events
BPIC18_D4e_D6b.xes: 24895 cases, 1384072 events
